In [ ]:
import json as _json

# Reutilizando a função de resumo do Nível 1
def montar_resumo_cliente(df, cliente_id):
    sub = df[df["cliente_id"] == cliente_id]
    return {
        "cliente_id": cliente_id,
        "qtd_operacoes": int(len(sub)),
        "volume_total_brl": float(sub["valor_brl"].sum()),
        "canais_usados": sub["canal"].value_counts().to_dict(),
        "contrapartes": sorted(sub["contraparte"].unique().tolist()),
        "operacoes_flagueadas": sub[sub["flag_valor_atipico"] | sub["flag_fracionamento"]][
            ["id", "data", "valor_brl", "canal", "tipo", "contraparte", "observacao"]
        ].to_dict("records"),
    }

# Função de processamento em lote
def processar_lote(df, lista_clientes):
    pareceres = {}
    for cliente in lista_clientes:
        resumo = montar_resumo_cliente(df, cliente)
        
        # PROMPT_V2 (o estruturado do Nível 1)
        prompt = f"""Você é analista sênior de PLD. Responda ESTRITAMENTE em JSON com as chaves:
        - "nivel_risco": "baixo", "medio" ou "alto"
        - "tipologia_suspeita": string curta
        - "red_flags": lista de strings
        - "justificativa": 2-4 frases
        
        Resumo do cliente: {resumo}
        """
        
        # Chama a LLM (mock=True ou mock=False configurado)
        resultado = chamar_llm(prompt, mock=True) # Mude para False para rodar de verdade
        
        # Valida e armazena
        parecer_json, erro = valida_parecer(resultado["texto"])
        if erro:
            pareceres[cliente] = {"erro": erro}
        else:
            pareceres[cliente] = parecer_json
            
    return pareceres

# Executa o lote para os suspeitos do Nível 2
resultados_lote = processar_lote(df, clientes_suspeitos)

# Exibe o resultado de um cliente como exemplo
print(_json.dumps(resultados_lote, indent=2, ensure_ascii=False))import json as _json

# Reutilizando a função de resumo do Nível 1
def montar_resumo_cliente(df, cliente_id):
    sub = df[df["cliente_id"] == cliente_id]
    return {
        "cliente_id": cliente_id,
        "qtd_operacoes": int(len(sub)),
        "volume_total_brl": float(sub["valor_brl"].sum()),
        "canais_usados": sub["canal"].value_counts().to_dict(),
        "contrapartes": sorted(sub["contraparte"].unique().tolist()),
        "operacoes_flagueadas": sub[sub["flag_valor_atipico"] | sub["flag_fracionamento"]][
            ["id", "data", "valor_brl", "canal", "tipo", "contraparte", "observacao"]
        ].to_dict("records"),
    }

# Função de processamento em lote
def processar_lote(df, lista_clientes):
    pareceres = {}
    for cliente in lista_clientes:
        resumo = montar_resumo_cliente(df, cliente)
        
        # PROMPT_V2 (o estruturado do Nível 1)
        prompt = f"""Você é analista sênior de PLD. Responda ESTRITAMENTE em JSON com as chaves:
        - "nivel_risco": "baixo", "medio" ou "alto"
        - "tipologia_suspeita": string curta
        - "red_flags": lista de strings
        - "justificativa": 2-4 frases
        
        Resumo do cliente: {resumo}
        """
        
        # Chama a LLM (mock=True ou mock=False configurado)
        resultado = chamar_llm(prompt, mock=True) # Mude para False para rodar de verdade
        
        # Valida e armazena
        parecer_json, erro = valida_parecer(resultado["texto"])
        if erro:
            pareceres[cliente] = {"erro": erro}
        else:
            pareceres[cliente] = parecer_json
            
    return pareceres

# Executa o lote para os suspeitos do Nível 2
resultados_lote = processar_lote(df, clientes_suspeitos)

# Exibe o resultado de um cliente como exemplo
print(_json.dumps(resultados_lote, indent=2, ensure_ascii=False))